In [1]:
import json
from pathlib import Path
import pandas as pd

from utils import load_data, NUMERIC_FEATURES_FULL, CATEGORICAL_FEATURES, PROJECT_ROOT

X_train, X_test, y_train, y_test, _, _ = load_data()

# Справочник строим по объединению train+test — иначе frontend не увидит станций,
# попавших только в test. Сам факт обучения от этого не страдает.
X_all = pd.concat([X_train, X_test], ignore_index=True)
print(f"Всего строк для построения справочника: {len(X_all):,}")


Всего строк для построения справочника: 11,024


In [2]:
categorical_catalog = {}

for col in CATEGORICAL_FEATURES:
    series = X_all[col].dropna().astype(str)
    value_counts = series.value_counts()
    categorical_catalog[col] = {
        "n_unique": int(series.nunique()),
        "values": sorted(value_counts.index.tolist()),  # отсортировано по алфавиту
        "value_counts": {k: int(v) for k, v in value_counts.items()},
    }
    print(f"{col}: {series.nunique()} уникальных значений")


metro_station: 295 уникальных значений
renovation: 4 уникальных значений


In [3]:
numeric_catalog = {}

for col in NUMERIC_FEATURES_FULL:
    s = X_all[col].dropna()
    numeric_catalog[col] = {
        "dtype": str(X_all[col].dtype),
        "min": float(s.min()),
        "max": float(s.max()),
        "median": float(s.median()),
        "mean": float(s.mean()),
        "q01": float(s.quantile(0.01)),
        "q99": float(s.quantile(0.99)),
    }

pd.DataFrame(numeric_catalog).T.round(2)


,dtype,min,max,median,mean,q01,q99
area,float64,11.0,396.5,60.0,80.320226,13.0,321.539
rooms,float64,0.0,12.0,2.0,2.339441,0.0,6.0
floor,float64,1.0,92.0,6.0,8.805878,1.0,39.0
total_floors,int64,1.0,97.0,15.0,17.548893,3.0,60.0
minutes_to_metro,float64,1.0,60.0,11.0,12.447206,2.0,34.0
floor_ratio,float64,0.025641,1.0,0.5,0.515815,0.058824,1.0
is_first_floor,int64,0.0,1.0,0.0,0.09307,0.0,1.0
is_last_floor,int64,0.0,1.0,0.0,0.081368,0.0,1.0
is_studio,int64,0.0,1.0,0.0,0.171716,0.0,1.0
log_area,float64,2.484907,5.985195,4.110874,4.138452,2.639057,5.776224


In [4]:
catalog = {
    "version": 1,
    "dataset_size": int(len(X_all)),
    "target": {
        "name": "price",
        "min": float(y_train.min()),
        "max": float(y_train.max()),
        "median": float(y_train.median()),
        "mean": float(y_train.mean()),
    },
    "numeric_features": numeric_catalog,
    "categorical_features": categorical_catalog,
}

catalog_path = PROJECT_ROOT / "ml_models" / "feature_catalog.json"
catalog_path.parent.mkdir(parents=True, exist_ok=True)

with open(catalog_path, "w", encoding="utf-8") as f:
    json.dump(catalog, f, ensure_ascii=False, indent=2)

print(f"Справочник сохранён: {catalog_path}")
print(f"Размер: {catalog_path.stat().st_size / 1024:.1f} КБ")


Справочник сохранён: C:\Users\roman\PycharmProjects\real_estate_ML\ml_models\feature_catalog.json
Размер: 24.2 КБ


In [5]:
with open(catalog_path, "r", encoding="utf-8") as f:
    loaded = json.load(f)

print("Варианты ремонта:", loaded["categorical_features"]["renovation"]["values"])
print(f"\nПервые 10 станций метро: {loaded['categorical_features']['metro_station']['values'][:10]}")
print(f"\nДиапазон area: {loaded['numeric_features']['area']['min']:.1f} – "
      f"{loaded['numeric_features']['area']['max']:.1f} м²")


Варианты ремонта: ['Cosmetic', 'Designer', 'European-style renovation', 'Without renovation']

Первые 10 станций метро: ['Авиамоторная', 'Автозаводская', 'Академическая', 'Александровский сад', 'Алексеевская', 'Алма-Атинская', 'Алтуфьево', 'Аминьевская', 'Андроновка', 'Аннино']

Диапазон area: 11.0 – 396.5 м²
